## Gaussian Mixture Model (GMM)

Gaussian Mixture Model (GMM) is a probabilistic clustering algorithm that assumes the data is generated from a mixture of multiple Gaussian distributions.

Unlike K-Means, which assigns each data point to a single cluster, GMM provides **soft cluster assignments**, meaning each data point has a probability of belonging to each cluster.

### How GMM Works

The algorithm uses an iterative process called the Expectation-Maximization (EM) algorithm:

1. **Initialize parameters** (means, covariances, and mixture weights)
2. **Expectation step (E-step)**:
   - Compute the probability that each data point belongs to each cluster
3. **Maximization step (M-step)**:
   - Update cluster parameters based on these probabilities
4. Repeat until convergence

### Objective Function

GMM aims to maximize the **likelihood of the data** under the mixture model:

- Each cluster is modeled as a Gaussian distribution
- The model finds parameters that best explain the observed data

### Key Characteristics

- Produces **soft probabilistic assignments**
- Can model:
  - overlapping clusters
  - elliptical cluster shapes (depending on covariance type)
- More flexible than K-Means

### Limitations

- Requires specifying the number of components **K**
- Sensitive to initialization
- Can be computationally more expensive than K-Means
- Assumes data follows Gaussian-like distributions

### Application in This Project

In this project, GMM is used to model the **overlapping structure** of fake and true news in embedding space.

Instead of forcing each article into a single cluster, GMM assigns a probability distribution over clusters. These probabilities are then used to construct a **fake risk score**, which reflects how strongly an article is associated with clusters dominated by fake news.

This approach better captures the **continuous and uncertain nature of misinformation**, where articles may not belong entirely to a single category.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --------------------------------------------------
# Evaluate different values of K for GMM
# --------------------------------------------------

# Use normalized embeddings for clustering

for k in [5, 8, 10, 12, 15]:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="diag",   # diagonal covariance for high-dimensional embeddings
        random_state=42
    )
    labels = gmm.fit_predict(X_embed_norm)

    print(f"\n=== k = {k} ===")

    # 1) Cluster sizes
    print("Cluster counts:")
    print(pd.Series(labels).value_counts())

    # 2) Fake proportion per cluster
    df_temp = df_sub.copy()
    df_temp["gmm_cluster"] = labels
    fake_props = (
        df_temp.groupby("gmm_cluster")["label"]
        .mean()
        .sort_values(ascending=False)
    )

    print("\nFake proportion per cluster:")
    print(fake_props)

    # 3) Coverage of fake articles in high-purity clusters
    # High-purity clusters are clusters where the mean fake label is > 0.8
    mask = df_temp.groupby("gmm_cluster")["label"].transform("mean") > 0.8
    coverage = df_temp[mask]["label"].sum() / df_temp["label"].sum()

    print(f"\nFake coverage (>80% fake clusters): {coverage:.3f}")
    print("-" * 50)

# --------------------------------------------------
# Fit final GMM with k = 10
# --------------------------------------------------

# I selected k = 10 because it provided a stable set of clusters while maintaining reasonable fake-news coverage in high-purity clusters. This makes it a better choice for constructing a cluster-based risk score than larger values of k, which produced smaller but purer clusters with lower coverage.
gmm = GaussianMixture(
    n_components=10,
    covariance_type="diag",
    random_state=42
)

labels_gmm = gmm.fit_predict(X_embed_norm)
probs_gmm = gmm.predict_proba(X_embed_norm)   # shape: (6000, 10)

# Compute fake rate for each cluster
df_sub = df_sub.copy()
df_sub["gmm_cluster"] = labels_gmm

cluster_fake_rate = (
    df_sub.groupby("gmm_cluster")["label"]
    .mean()                                  # average fake proportion in each cluster
    .reindex(range(gmm.n_components))        # keep cluster order consistent
    .values
)

print("Cluster fake rates:")
print(cluster_fake_rate)

# Example interpretation:
# If an article belongs to cluster 9, that cluster contains a high proportion of fake articles.

# --------------------------------------------------
# Convert soft cluster membership into a continuous risk score
# --------------------------------------------------
# Convert to Risk Score
# Rather than assigning each article to a single cluster, I compute a weighted risk score based on soft cluster membership,
# which better captures the continuous and overlapping nature of semantic space.
risk_score = probs_gmm @ cluster_fake_rate
df_sub["fake_risk_score"] = risk_score

print(df_sub.groupby("label")["fake_risk_score"].mean())
# label
# 0    0.417456
# 1    0.657638
# True articles are on average, assigned moderate fake risk
# Fake articles are on average, assigned higher risk

# Visualization of the distributions
plt.hist(df_sub[df_sub["label"] == 0]["fake_risk_score"], bins=30, alpha=0.5, label="True")
plt.hist(df_sub[df_sub["label"] == 1]["fake_risk_score"], bins=30, alpha=0.5, label="Fake")
plt.legend()
plt.title("Risk Score Distribution")
plt.show()
# X-axis = fake risk score (0:1)
# Y-axis = number of articles
# Brown region is the overlapping between fake and true distributions
# The plot is showing that fake news tend to receive higher risk score than true articles.
# So when users sees ~0.1:0.4 this represent mostly true article. 0.5:0.7 middle range.

# Compute ROC AUC:
auc = roc_auc_score(df_sub["label"], df_sub["fake_risk_score"])
print("AUC:", auc)
# AUC: 0.775452 (GMM model ranks fake articles higher than true ones 77.5% of the time

# Plot ROC Curve
y_true = df_sub["label"]
y_score = df_sub["fake_risk_score"]

fpr, tpr, thresholds = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], linestyle='--', color='gray')  # random baseline

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

### Comparison of Clustering Methods

- **K-Means** reveals stable partitions but imposes structure on the data
- **HDBSCAN** shows that the data does not form natural density-based clusters
- **GMM** captures overlapping and probabilistic structure

Together, these methods suggest that fake and true news are not perfectly separable, but instead exist along a continuous semantic spectrum.

However, supervised models can still achieve high performance because they do not rely on discovering natural clusters. Instead, they learn decision boundaries that exploit subtle but consistent patterns in the data, such as differences in writing style, topic distribution, or source characteristics.

### Key Insight

High classification accuracy does not imply true semantic separation, but rather reflects the model’s ability to leverage underlying patterns and dataset-specific signals.